In [ ]:
import numpy as np


delegations = np.array(([
    [1, 0, 0],
    [0, 1, 0],
    [0.3, 0.3, 0.4]
]))

delegations.mean(axis=0) * delegations.shape[0]

array([0.43333333, 0.43333333, 0.13333333])

In [94]:
import numpy as np

def softmax(x, axis=-1, T=1.0):
    e = np.exp((x / T) - np.max(x / T, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def simulate(
        class_accuracy: np.ndarray,
        deleg_precision: np.ndarray,
        n_elections = 1_000,
        N: int = 1_000,
        T: float= 1.0
    ):
    

    n_delegators = len(deleg_precision)
    n_predictors = len(class_accuracy)

    class_accuracy = class_accuracy[np.newaxis, :]
    deleg_precision = deleg_precision[:, np.newaxis]
    deleg_imprecision = 1 - deleg_precision 

    accuracies_avg = []
    accuracies_fixed = []
    for _ in range(N):
    
        noisy_accuracy_estimate = np.random.uniform(size=(n_delegators, n_predictors)) 
        accuracy_estimate = deleg_precision * class_accuracy + deleg_imprecision * noisy_accuracy_estimate
        accuracy_estimate = np.maximum(accuracy_estimate, 0.5)
        accuracy_estimate_basline = (accuracy_estimate - 0.5) * 2
        delegations = accuracy_estimate_basline / (np.sum(accuracy_estimate_basline, axis=-1, keepdims=True) + 1e-5)
        delegations = softmax(delegations, T=T)

        M = delegations

        w_avg = M.mean(axis=0)
        w_fixed = np.linalg.matrix_power(M, 100).sum(axis=0) / M.shape[0]


        votes = np.where(np.random.rand(n_elections, n_predictors) < class_accuracy, 1, -1)
        
        tally_avg = votes @ w_avg
        tally_fixed = votes @ w_fixed

        accuracies_avg.append(np.mean(tally_avg > 0))
        accuracies_fixed.append(np.mean(tally_fixed > 0))
        
    # print(f"Mean Accuracy (Avg):   {np.mean(accuracies_avg):.5f}")
    # print(f"Mean Accuracy (Fixed): {np.mean(accuracies_fixed):.5f}")
    
    return np.mean(accuracies_avg) - np.mean(accuracies_fixed)


    

accs = np.array([0.95, 0.5, 0.5, 0.5])
dela = np.array([0.95, 0.05, 0.05, 0.05])

simulate(accs, dela, T=0.01)


np.float64(-0.04503999999999997)

In [97]:
import numpy as np

def softmax(x, axis=-1, T=1.0):
    x_scaled = x / T
    e = np.exp(x_scaled - np.max(x_scaled, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def simulate_fast(
        class_accuracy: np.ndarray,
        deleg_precision: np.ndarray,
        n_elections=1_000,
        N: int=1_000,
        T: float=1.0
    ):
    
    n_delegators = len(deleg_precision)
    n_predictors = len(class_accuracy)

    # Reshape for 3D broadcasting: (Batch N, Delegators, Predictors)
    ca = class_accuracy.reshape(1, 1, n_predictors)
    dp = deleg_precision.reshape(1, n_delegators, 1)
    di = 1 - dp

    # 1. Vectorized Matrix (M) Generation
    noise = np.random.uniform(size=(N, n_delegators, n_predictors)) 
    acc_est = np.maximum(dp * ca + di * noise, 0.5)
    
    acc_est_baseline = (acc_est - 0.5) * 2
    delegations = acc_est_baseline / (np.sum(acc_est_baseline, axis=-1, keepdims=True) + 1e-5)
    M = softmax(delegations, axis=-1, T=T) # Shape: (N, n_delegators, n_predictors)

    # 2. Vectorized Weights Calculation
    w_avg = M.mean(axis=1) # Shape: (N, n_predictors)
    # matrix_power automatically handles batches of matrices
    w_fixed = np.linalg.matrix_power(M, 100).sum(axis=1) / n_delegators 

    # 3. Vectorized Votes and Tallies
    votes = np.where(np.random.rand(N, n_elections, n_predictors) < ca, 1, -1)
    
    # Batched matrix multiplication: (N, n_elections, n_predictors) @ (N, n_predictors, 1)
    tally_avg = votes @ np.expand_dims(w_avg, axis=-1)
    tally_fixed = votes @ np.expand_dims(w_fixed, axis=-1)

    return np.mean(tally_avg > 0) - np.mean(tally_fixed > 0)

In [105]:
from scipy.optimize import dual_annealing
import numpy as np

n_elements = 10

def objective(x):
    return simulate_fast(
        x[:n_elements], 
        x[n_elements:], 
        n_elections=200, 
        N=50, 
        T=0.1
    )

bounds = [(0.6, 1.0)] * n_elements + [(0.5, 1.0)] * n_elements
result = dual_annealing(objective, bounds, maxiter=1000)

print(result.x[:n_elements], result.x[n_elements:], result.fun)

[0.67457464 0.99621866 0.62103917 0.63253741 0.63154105 0.60261887
 0.63992577 0.64421244 0.610998   0.66329031] [0.56664398 0.9499115  0.509116   0.53579986 0.62537209 0.52400842
 0.56889479 0.99153367 0.51302048 0.50153413] -0.01550000000000007


In [61]:
np.repeat([[1, 3, 3]], 3, axis=0)

array([[1, 3, 3],
       [1, 3, 3],
       [1, 3, 3]])

In [1]:
import numpy as np

# Initial voting weight vector (v). 
# Index 0: Alice (10,000 votes)
# Index 1: Bob (1 vote)
# Index 2: Charlie (1 vote)
v = np.array([10_000, 1, 1])
total_votes = np.sum(v)

# ---------------------------------------------------------
# Algorithm 2: The Linear System (Fixed Point)
# ---------------------------------------------------------
# Delegation matrix M where M[i, j] = 1 means j delegates to i.
# Alice (0) and Bob (1) delegate to Charlie (2).
M = np.array([
    [0, 0, 0],  # No one delegates to Alice
    [0, 0, 0],  # No one delegates to Bob
    [1, 1, 0]   # Alice and Bob delegate to Charlie
])

# Solve: p = v + M*p  =>  p = (I - M)^(-1) * v
I = np.eye(3)
p_linear = np.linalg.inv(I - M).dot(v)

# ---------------------------------------------------------
# Algorithm 1: Average & Normalize
# ---------------------------------------------------------
# Averaging matrix A. 
# Charlie has 2 incoming links, so averaging treats them as 0.5 each.
A = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.5, 0.5, 0.0]
])

# Solve: p_raw = v + A*p_raw  =>  p_raw = (I - A)^(-1) * v
p_raw = np.linalg.inv(I - A).dot(v)

# Normalize to force the system to maintain the exact total volume of power (10,002)
normalization_factor = total_votes / np.sum(p_raw)
p_averaged = p_raw * normalization_factor

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------
print(f"--- TOTAL VOTES IN SYSTEM: {total_votes} ---\n")

print("ALGORITHM 2: LINEAR SYSTEM (The correct epistemic signal)")
print(f"Alice:   {p_linear[0]:.1f}")
print(f"Bob:     {p_linear[1]:.1f}")
print(f"Charlie: {p_linear[2]:.1f}")
print("Result: Charlie correctly inherits 10,000 from Alice + 1 from Bob + his own 1.\n")

print("ALGORITHM 1: AVERAGE & NORMALIZE (The corrupted signal)")
print(f"Alice:   {p_averaged[0]:.1f}")
print(f"Bob:     {p_averaged[1]:.1f}")
print(f"Charlie: {p_averaged[2]:.1f}")
print("Result: Alice's 10,000 votes were mathematically crushed by Bob's 1 vote.")

--- TOTAL VOTES IN SYSTEM: 10002 ---

ALGORITHM 2: LINEAR SYSTEM (The correct epistemic signal)
Alice:   10000.0
Bob:     1.0
Charlie: 10002.0
Result: Charlie correctly inherits 10,000 from Alice + 1 from Bob + his own 1.

ALGORITHM 1: AVERAGE & NORMALIZE (The corrupted signal)
Alice:   6666.9
Bob:     0.7
Charlie: 3334.4
Result: Alice's 10,000 votes were mathematically crushed by Bob's 1 vote.
